In [1]:
import numpy as np
import pandas as pd
from scipy.integrate import solve_ivp
import seaborn as sns
import matplotlib.pyplot as plt
import os
import pickle as pkl
import json

In [2]:
#df_Nfd = pd.read_csv('/home/apoorva/Desktop/work/MZ_analysis/data/total_mzcounts_sorted.csv')
df_new = pd.read_csv('/home/apoorva/Desktop/work/MZ_analysis/data/new_data_MZB.csv')
df_ont = pd.read_csv('/home/apoorva/Desktop/work/MZ_analysis/data/New_data/ontogeny_counts.csv')

print(df_ont.head(10))

                              mouse  age.at.S1K      MZ_Ki67+  \
0    Samples_Spleen_002_d11_017.fcs          11   2750.222222   
1    Samples_Spleen_003_d11_018.fcs          11   1000.000000   
2    Samples_Spleen_004_d11_019.fcs          11   2849.805310   
3    Samples_Spleen_005_d11_020.fcs          11    269.856985   
4    Samples_Spleen_006_d14_021.fcs          14  56055.042735   
5    Samples_Spleen_007_d14_022.fcs          14  27720.194489   
6    Samples_Spleen_008_d14_023.fcs          14    753.412564   
7    Samples_Spleen_009_d14_024.fcs          14  32426.329114   
8    Samples_Spleen_010_d14_025.fcs          14  45418.865979   
9  X20181210_Spleen_d18_002_055.fcs          18   3931.767372   

   T1 AA4.1+_Ki67+       MZ_total  T1 AA4.1+_total  frac_MZ_Ki67+  \
0     2.766195e+06   12058.666667     4.145272e+06       0.228070   
1     1.983750e+06    7450.000000     3.223400e+06       0.134228   
2     2.222848e+06   18873.238938     3.302817e+06       0.150997   
3     2.

In [3]:

# Change column name to match the other dataframe
df_new1 = df_new.rename(columns = {'Age at S1K':'age.at.S1K'})
df_ont1 = df_ont.rename(columns = {'T1 AA4.1+_total':'total_T1_AA4.1+', 'MZ_total':'total_MZ'})

df_T1counts_Nfd = df_new1.filter(items=['age.at.S1K', 'total_T1_AA4.1+', 'total_MZ'])
df_T1counts_ont = df_ont1.filter(items=['age.at.S1K', 'total_T1_AA4.1+', 'total_MZ'])

df_T1counts_ont.to_csv('/home/apoorva/Desktop/work/MZ_analysis/data/Ontogeny_T1.csv', index=False)

# print(df_T1counts_Nfd.head(10))
# print(df_T1counts_ont.head(10))

# combine in one dataframe
total_T1counts = pd.concat([df_T1counts_Nfd, df_T1counts_ont])
total_T1counts = total_T1counts.sort_values(by='age.at.S1K')
print(total_T1counts.head(10))

# Creating a list to use as an input in Stan model

# data = {'numobs': len(total_T1counts['age.at.S1K']), 
#         'numsolve' : total_T1counts['age.at.S1K'].tolist(),
#         'total_counts' : total_T1counts['total_T1_AA4.1+'].tolist(),
#         'time_pred' : np.array(np.arange(10,731,1)).tolist(),
#         'num_pred' : len(np.array(np.arange(10,731,1)))
# }

# json_string = json.dumps(data)

# file_path1 = '/home/apoorva/Desktop/work/MZ_analysis/data/New_data/data_T1fits.json'
# with open(file_path1, 'wb') as json_file:
#     json_file.write(json_string.encode())

   age.at.S1K  total_T1_AA4.1+       total_MZ
2          11     3.302817e+06   18873.238938
3          11     2.677656e+06   11941.171604
1          11     3.223400e+06    7450.000000
0          11     4.145272e+06   12058.666667
6          14     9.934749e+06   55375.823430
7          14     9.792409e+06  102645.316456
4          14     6.970682e+06  173580.615385
5          14     8.923469e+06   65914.813614
8          14     8.129739e+06  152588.316151
9          18     8.048254e+06  120475.287009


In [4]:
# data time points
data_time = total_T1counts['age.at.S1K'].values
# data_time = data_time[4:]

# Create a dataframe with unique age at S1K values
dfunique = total_T1counts.drop_duplicates(subset='age.at.S1K', keep='first')
dfunique = dfunique.sort_values(by='age.at.S1K')
dfunique = dfunique.reset_index(drop=True)

# unique time points in data
solve_time = dfunique['age.at.S1K'].values
# solve_time = solve_time[1:]
# solve_ageatbmt = dfunique['Age at BMT'].values

# create index map of data time to solve time for Nfd
time_index_map = np.zeros(len(data_time), dtype=int)
for i, t in enumerate(solve_time):
    time_index_map[data_time == t] = i+1

print(time_index_map)
#time sequence for predictions specific to agebins within the data
time_pred1_solve= np.array(np.arange(11, 732, 0.1))
# time_pred3_solve= np.array(np.arange(88, 750, 1))
# time_pred2_solve= np.array(np.arange(66, 750, 1))
# time_pred1_initial = 11.0
# time_pred3_bmt= 87.0
# time_pred2_bmt= 65.0

[ 1  1  1  1  2  2  2  2  2  3  3  3  3  4  4  5  5  5  5  5  5  6  6  6
  6  7  7  7  7  8  8  8  8  8  8  9  9 10 10 10 10 10 11 12 12 13 13 14
 14 14 14 15 15 16 17 18 19 20 21 22 23 24 25 26 26 27 28 29 30 31 32 33
 34 35 36 36 37 38 39 40 41 41 42 42]


In [5]:
# Create a dataframe with unique age at S1K values for ontogeny data
# data time points
data_time_ont = df_T1counts_ont['age.at.S1K'].values
# data_time_ont = data_time_ont[4:]

# Create a dataframe with unique age at S1K values
df_unique_ont = df_T1counts_ont.drop_duplicates(subset='age.at.S1K', keep='first')
df_unique_ont = df_unique_ont.sort_values(by='age.at.S1K')
df_unique_ont = df_unique_ont.reset_index(drop=True)

# unique time points in data
solve_time_ont = df_unique_ont['age.at.S1K'].values
# solve_time_ont = solve_time_ont[1:]
# solve_ageatbmt = dfunique['Age at BMT'].values

# create index map of data time to solve time for Nfd
time_index_map_ont = np.zeros(len(data_time_ont), dtype=int)
for i, t in enumerate(solve_time_ont):
    time_index_map_ont[data_time_ont == t] = i+1

print(len(time_index_map_ont))
print(solve_time_ont)
print(data_time_ont)
print(len(np.array(df_T1counts_ont['total_MZ'])))
#time sequence for predictions specific to agebins within the data
time_pred_solve= np.array(np.arange(11, 732, 0.1))

62
[ 11  14  18  19  20  22  23  24  25  26  27  28  29  31  33  36  39  43
  53  57  71 113 152]
[ 11  11  11  11  14  14  14  14  14  18  18  18  18  19  19  20  20  20
  20  20  20  22  22  22  22  23  23  23  23  24  24  24  24  24  24  25
  25  26  26  26  26  26  27  28  28  29  29  31  31  31  31  33  33  36
  39  43  53  57  71 113 152 152]
62


In [6]:
data_mz_ontogenyT1={"numobs" : len(data_time_ont),  # Number of donor fraction observations
      "numsolve" : len(solve_time_ont),  # Number of unique time points in the data
      "data_time" : data_time_ont,
      "time_index_map" : time_index_map_ont,
      "solve_time" : solve_time_ont,
    #   "totalMZ" : np.array(df_T1counts_ont['total_MZ'][4:]),
      "totalMZ" : np.array(df_T1counts_ont['total_MZ']),
      
      "time_pred1_solve" : time_pred_solve,
      "numpred" : len(time_pred_solve)}

# Export data to JSON
def numpy_array_encoder(obj):
    if isinstance(obj, np.ndarray):
        return obj.tolist()  # Convert numpy array to a list
    raise TypeError("Object of type {} is not JSON serializable".format(type(obj)))

# Serialize dictionary to JSON using custom encoder function
json_string = json.dumps(data_mz_ontogenyT1, default=numpy_array_encoder)

file_path= "/home/apoorva/Desktop/work/MZ_analysis/data/MZcounts_ontogeny_T1.json"
with open(file_path, "w") as json_file:  json_file.write(json_string)

print("Data saved to", file_path)

Data saved to /home/apoorva/Desktop/work/MZ_analysis/data/MZcounts_ontogeny_T1.json


In [7]:
#Binning the data for Nfd dataframe on equally sized aged bins based on age at BMT
from numpy import size

print(data_time)
print(solve_time)

[ 11  11  11  11  14  14  14  14  14  18  18  18  18  19  19  20  20  20
  20  20  20  22  22  22  22  23  23  23  23  24  24  24  24  24  24  25
  25  26  26  26  26  26  27  28  28  29  29  31  31  31  31  33  33  36
  39  43  53  57  59  62  69  71  76  88  88  92  95 102 109 113 119 123
 124 141 152 152 158 212 219 291 306 306 731 731]
[ 11  14  18  19  20  22  23  24  25  26  27  28  29  31  33  36  39  43
  53  57  59  62  69  71  76  88  92  95 102 109 113 119 123 124 141 152
 158 212 219 291 306 731]


In [8]:
# Creating a list to use as an input in Stan model

data_mz_ontogenyT1={"numobs" : len(data_time),  # Number of donor fraction observations
      "numsolve" : len(solve_time),  # Number of unique time points in the data
      "data_time" : data_time,
      "time_index_map" : time_index_map,
      "solve_time" : solve_time,
      "totalMZ" : np.array(total_T1counts['total_MZ'][4:]),
      "time_pred1_solve" : time_pred1_solve,
      "numpred" : len(time_pred1_solve)}

# Export data to JSON
def numpy_array_encoder(obj):
    if isinstance(obj, np.ndarray):
        return obj.tolist()  # Convert numpy array to a list
    raise TypeError("Object of type {} is not JSON serializable".format(type(obj)))

# Serialize dictionary to JSON using custom encoder function
json_string = json.dumps(data_mz_ontogenyT1, default=numpy_array_encoder)

file_path= "/home/apoorva/Desktop/work/MZ_analysis/data/MZBcounts_allages.json"
with open(file_path, "w") as json_file:  json_file.write(json_string)

print("Data saved to", file_path)
  

Data saved to /home/apoorva/Desktop/work/MZ_analysis/data/MZBcounts_allages.json
